# 문제 2-3 : 학생 정보 구조화 시스템

학생의 자유 형식 자기소개를 입력받아 **이름, 나이, 전공, 취미 리스트, 목표**를 구조화된 형태로 추출한다.

**출력 파서**: `PydanticOutputParser`

**요구사항 체크**
- [x] `PydanticOutputParser` 와 `BaseModel` 사용
- [x] 각 필드에 적절한 타입(`str`, `int`, `list[str]`)과 설명(`Field(description=...)`) 추가
- [x] 자유 형식의 텍스트에서 정보 추출 (문제 예시 + 말투·순서가 다른 자기소개 2개)

## 1. 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정 (.env 는 프로젝트 루트에 있음)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 전체는 출력하지 않고 설정 여부만 확인
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

## 2. LLM 설정 (ChatOpenAI)
정보 추출 작업이라 답변이 흔들리지 않도록 `temperature=0` 으로 설정한다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq"(무료) 또는 "openai" — 이 값만 바꾸면 아래 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# Groq는 OpenAI 호환 API라서 ChatOpenAI에 base_url만 바꿔서 사용
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}


def get_llm(temperature: float = 0.7, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자의 ChatOpenAI 인스턴스를 생성합니다."""
    config = LLM_CONFIGS[provider or PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {(provider or PROVIDER).upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, max_retries=3)


llm = get_llm(temperature=0)
print(f"{PROVIDER} / {llm.model_name}")

## 3. 데이터 모델 정의 — `BaseModel`
`Field(description=...)` 의 설명은 파서가 만드는 JSON 스키마에 그대로 들어가서, LLM 이 각 필드에 무엇을 채워야 하는지 알려 주는 역할을 한다.

In [ ]:
from pydantic import BaseModel, Field


class StudentInfo(BaseModel):
    """학생 자기소개에서 추출한 정보"""
    name: str = Field(description="학생의 이름 (예: 김민수)")
    age: int = Field(description="학생의 나이, 숫자만 (예: 22)")
    major: str = Field(description="전공 (예: 컴퓨터공학)")
    hobbies: list[str] = Field(description="취미 목록, 취미 하나당 원소 하나 (예: ['게임하기', '영화보기'])")
    goal: str = Field(description="앞으로의 목표 (예: 훌륭한 개발자가 되는 것)")

## 4. 출력 파서 생성 — `PydanticOutputParser`
- `get_format_instructions()` : `StudentInfo` 의 JSON 스키마를 담은 형식 지시문을 만든다
- `parse()` : LLM 이 준 JSON 을 `StudentInfo` 객체로 바꾸고, 타입 검증까지 한다 (예: `"22"` → `22`)

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=StudentInfo)

format_instructions = output_parser.get_format_instructions()
print(format_instructions)

## 5. PromptTemplate 작성 & LCEL 체인 연결: `prompt | llm | output_parser`

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """당신은 학생의 자기소개에서 핵심 정보를 정확하게 추출하는 도우미입니다.
아래 자기소개를 읽고 이름, 나이, 전공, 취미 목록, 목표를 추출하세요.
- 나이는 숫자(정수)로 변환 (예: '스물두 살' → 22)
- 취미는 하나씩 나누어 리스트로 작성
- 자기소개에 있는 표현을 최대한 그대로 사용

자기소개:
{introduction}

{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["introduction"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm | output_parser

## 6. 실행 — 문제의 예시 입력

In [ ]:
import json

example_intro = (
    "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. "
    "취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."
)

student = chain.invoke({"introduction": example_intro})

print(type(student))  # dict 가 아니라 StudentInfo(BaseModel) 객체
print(json.dumps(student.model_dump(), ensure_ascii=False, indent=4))

`BaseModel` 객체라서 속성으로 바로 접근할 수 있고, 선언한 타입대로 변환되어 있다.

In [ ]:
print(f"이름 : {student.name}")
print(f"나이 : {student.age}  (type: {type(student.age).__name__})")
print(f"전공 : {student.major}")
print(f"취미 : {', '.join(student.hobbies)}  (type: {type(student.hobbies).__name__}, {len(student.hobbies)}개)")
print(f"목표 : {student.goal}")

## 7. 다른 형식의 자기소개로 테스트
말투, 정보 순서, 나이 표기(한글 숫자)가 다른 자유 형식 텍스트에서도 같은 구조로 추출되는지 확인한다.

In [ ]:
other_intros = [
    "반갑습니다~ 경영학과 다니는 이서연이라고 해요. 나이는 스물한 살이에요! "
    "주말마다 등산을 다니고, 카페 투어랑 사진 찍는 것도 좋아해요. 졸업하면 마케팅 전문가로 일하고 싶어요.",
    "제 이름은 박준호, 올해 25살. 전공은 전자공학입니다. "
    "기타 연주와 축구가 취미고, 반도체 설계 엔지니어가 되는 게 꿈입니다.",
]

for i, intro in enumerate(other_intros, 1):
    info = chain.invoke({"introduction": intro})
    print(f"[입력 {i}] {intro}")
    print(json.dumps(info.model_dump(), ensure_ascii=False, indent=4))
    print("-" * 70)

## 8. (선택) 직접 자기소개 입력해 보기

In [ ]:
introduction = input("자기소개를 입력하세요: ").strip()

if introduction:
    info = chain.invoke({"introduction": introduction})
    print(json.dumps(info.model_dump(), ensure_ascii=False, indent=4))
else:
    print("자기소개가 입력되지 않았습니다.")